# 2025년도 빅데이터와 인공지능을 활용한 시스템 강건설계 데이터 챌린지!

#### - 데이터 챌린지 목표 
: 빅데이터 핸들링 능력, 데이터 특징 추출 및 선택, 차원축소 및 시각화 수행능력 평가
#### - 제공 데이터 구성 : 센서데이터(정상/고장1/고장2 각 10개) + 기록데이터(정상/고장1/고장2 각 1개)
#### - 총 4단계의 데이터 챌린지를 수행하며, 단계별 결과가 저장된 폴더(Result)와 최종 코드파일(.ipynb)을 1개 압축파일(.zip)로 제출
#### +++ csv 데이터 읽을 때 header=None 옵션 사용필수
#### +++ csv 데이터 저장할 때 header=None, index=None 옵션 사용필수
#### +++ 해당 파일의 Help Code 활용하지 않아도 결과만 맞으면 무관

### ! 과제 수행 전 확인사항 !   
이 코드 파일과 Data, SplittedData, Result 폴더의 경로는 AI_Code/DataChallenge 폴더 안에 위치시킬 것

.

.

.

## 라이브러리 import

In [ ]:
import pandas as pd
import numpy  as np
import scipy.stats as sp
import pywt
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## 센서데이터, 기록데이터 보기

### - 센서데이터 :
* 정상, 고장1(Shunting Effect), 고장2(전극 비정렬) 각각 10개 센서데이터
* 전류 / 전압 / 가속도 3열로 구성(시간열 없음 주의!)
* Sampling Frequency: 12800Hz
* Sampling Time: 35초
* 데이터 1개당 12개의 용접 스폿이 포함됨

### - 기록데이터 :
* 정상, 고장1(Shunting Effect), 고장2(전극 비정렬) 각각 1개 기록데이터
* 기록데이터의 각 행: 용접 시작 시점의 인덱스 12개

### 데이터 살펴보기

In [ ]:
SensorData = pd.read_csv('./Data/Normal/Data_1.csv', header = None, names = ['Current', 'Voltage', 'Acceleration']) # 센서 데이터
RecordData = pd.read_csv('./Data/Normal/RecordData.csv', header = None) # 기록 데이터

In [ ]:
SensorData

In [ ]:
plt.figure(1, figsize=(15,10))
# 전류데이터 Plot
plt.subplot(3,1,1)
plt.plot(SensorData.iloc[:,0], c = 'r', label = SensorData.columns[0])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
# 전압데이터 Plot
plt.subplot(3,1,2)
plt.plot(SensorData.iloc[:,1], c = 'g', label = SensorData.columns[1])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
# 가속도데이터 Plot
plt.subplot(3,1,3)
plt.plot(SensorData.iloc[:,2], c = 'b', label = SensorData.columns[2])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
plt.show()

In [ ]:
RecordData

In [ ]:
# 정상 스폿용접 센서 데이터 보기
DataNo = 1   # 1~10
SpotNo = 10  # 1~12

SensorData = pd.read_csv('./Data/Normal/Data_%d.csv'%DataNo, header = None, names = ['Current', 'Voltage', 'Acceleration']) # 센서 데이터

start_index = RecordData.iloc[DataNo - 1, SpotNo - 1]
SensorData_spot = SensorData.iloc[start_index:start_index+2774, :]

plt.figure(1, figsize=(15,10))
# 전류데이터 Plot
plt.subplot(3,1,1)
plt.plot(SensorData_spot.iloc[:,0], c = 'r', label = SensorData_spot.columns[0])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
# 전압데이터 Plot
plt.subplot(3,1,2)
plt.plot(SensorData_spot.iloc[:,1], c = 'g', label = SensorData_spot.columns[1])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
# 가속도데이터 Plot
plt.subplot(3,1,3)
plt.plot(SensorData_spot.iloc[:,2], c = 'b', label = SensorData_spot.columns[2])
plt.legend(loc = 'upper right', fontsize = 12)
plt.grid()
plt.show()

.

.

.

# [1단계] 센서데이터 분할
> #### 기록데이터 상 각각의 용접 시작 시점부터 2774행만큼(≒0.2167초간) 해당 센서데이터 Spot별로 분할, 저장
> #### 저장 경로 : 이 코드 파일이 위치한 경로의 ' SplittedData' 폴더 내부

## 필수!

#### SplittedData 폴더 내부 각 데이터 파일명:
* 정상(Normal) : Normal_1.csv, Normal_2.csv, Normal_3.csv, ..., Normal_120.csv
* 고장1(Fault Mode 1) : Abnormal1_1.csv, Abnormal1_2.csv, Abnormal1_3.csv, ..., Abnormal1_120.csv
* 고장2(Fault Mode 2) : Abnormal2_1.csv, Abnormal2_2.csv, Abnormal2_3.csv, ..., Abnormal2_120.csv

In [ ]:
# Guide Code : 활용하지 않아도 결과만 맞으면 무관

NoOfData = 10
NoOfSpot = 12

RecordData_Normal    = pd.read_csv('./Data/Normal/RecordData.csv'   , header = None) # 정상  기록데이터 불러오기
RecordData_Abnormal1 = pd.read_csv('./Data/Abnormal1/RecordData.csv', header = None) # 고장1 기록데이터 불러오기
RecordData_Abnormal2 = pd.read_csv('./Data/Abnormal2/RecordData.csv', header = None) # 고장2 기록데이터 불러오기

for i in range(NoOfData):
    temp_path1 = './Data/Normal/Data_%d.csv'%(i+1)     # 정상  데이터 불러오기
    temp_path2 = './Data/Abnormal1/Data_%d.csv'%(i+1)  # 고장1 데이터 불러오기
    temp_path3 = './Data/Abnormal2/Data_%d.csv'%(i+1)  # 고장2 데이터 불러오기

    ??
    ??
    ??
    
    for j in range(NoOfSpot):
        ??
        ??
        ??
        
        
        temp_Normal.to_csv('./SplittedData/Normal_%d.csv'%(i*NoOfSpot+j+1), header=None, index=None)
        temp_Abnormal1.to_csv('./SplittedData/Abnormal1_%d.csv'%(i*NoOfSpot+j+1), header=None, index=None)
        temp_Abnormal2.to_csv('./SplittedData/Abnormal2_%d.csv'%(i*NoOfSpot+j+1), header=None, index=None)

### !!! (수강생 번호 외 코드 수정 X) 1단계 결과물을 제출용 파일로 저장

In [ ]:
StudentNo = 0   # 수강생 번호 입력


Step1File1 = pd.read_csv('./SplittedData/Normal_120.csv', header=None)
Step1File2 = pd.read_csv('./SplittedData/Abnormal1_120.csv', header=None)
Step1File3 = pd.read_csv('./SplittedData/Abnormal2_120.csv', header=None)

Path1 = './Result/ST%d_DC1_1.csv'%StudentNo
Path2 = './Result/ST%d_DC1_2.csv'%StudentNo
Path3 = './Result/ST%d_DC1_3.csv'%StudentNo

Step1File1.to_csv(Path1, header=None, index=None)
Step1File2.to_csv(Path2, header=None, index=None)
Step1File3.to_csv(Path3, header=None, index=None)

.

.

.

# [2단계] Time, Frequency domain 특징 추출

> ### 1단계에서 분할한 데이터들을 대상으로 정상/고장1/고장2 데이터들을 모두 불러와서 특징 추출하기 

## 필수!
* 특징값 종류 및 순서(실습코드와 동일) : Max, Min, Mean, RMS, Variance, Skewness, Kurtosis, Crest factor, Shape factor, Impulse factor
* 웨이블릿 분해 : MotherWavelet = haar, Level = 8로 설정
* 데이터당 시간 영역 특징 30개, 주파수 영역(웨이블릿 분해) 특징 240개 추출 => 전체 특징데이터 Shape (270, 360)
* 추출한 특징데이터(DataFrame) 변수명: FeatureData

In [ ]:
NoOfData    = 120  # 정상/고장1/고장2 스폿용접 데이터 각 120개씩 
NoOfSensor  = 3    # 전류(Current), 전압(Voltage), 가속도(Acceleration)
NoOfFeature = 10   # 특징 개수:10개 (순서: Max, Min, Mean, RMS, Variance, Skewness, Kurtosis, Crest factor, Shape factor, Impulse factor)

#### 특징 추출(Time domain)

In [ ]:
def rms(x): # RMS 함수 정의
    return np.sqrt(np.mean(x**2))

In [ ]:
# Time Domain 특징값 추출

# 특징데이터 크기 지정
TimeFeature_Normal = np.zeros((NoOfSensor*NoOfFeature , NoOfData))
TimeFeature_Abnormal1 = np.zeros((NoOfSensor*NoOfFeature , NoOfData))
TimeFeature_Abnormal2 = np.zeros((NoOfSensor*NoOfFeature , NoOfData))

for i in range(NoOfData):
    
    # 데이터 불러오기
    temp_path1 = './SplittedData/Normal_%d.csv'%(i+1)      # 정상 데이터 파일 경로
    temp_path2 = './SplittedData/Abnormal1_%d.csv'%(i+1)   # 고장1 데이터 파일 경로
    temp_path3 = './SplittedData/Abnormal2_%d.csv'%(i+1)   # 고장2 데이터 파일 경로
    
    
    
    
    
    
    for j in range(NoOfSensor)
        # Normal Time Domain Feature
        
        
        
        
        
        
        
        
        
        
        
        # Abnormal1 Time Domain Feature
        
        
        
        
        
        
        
        
        
        
        
        # Abnormal2 Time Domain Feature
        
        
        
        
        
        
        
        
        
        
        


In [ ]:
# 시간영역 특징 합치기(가로 방향)



#### 특징 추출(Frequency domain)

In [ ]:
#Frequency Domain 특징값 추출 (Wavelet Transform 기반)
# Wavelet options

MotherWavelet = pywt.Wavelet('haar')   # Mother wavelet (모함수) 지정
Level   = 8                            # Wavelet 분해 레벨 지정
select  = 8                            # 특징추출 영역 고주파 영역부터 개수 지정 (d1~)

#Frequency Domain 특징값 추출 (Wavelet Transform 기반)
FreqFeature_Normal    = np.zeros(shape=(NoOfSensor*NoOfFeature*select , NoOfData))
FreqFeature_Abnormal1 = np.zeros(shape=(NoOfSensor*NoOfFeature*select , NoOfData))
FreqFeature_Abnormal2 = np.zeros(shape=(NoOfSensor*NoOfFeature*select , NoOfData))

for i in range(NoOfData):
    
    # 데이터 불러오기
    temp_path1 = './SplittedData/Normal_%d.csv'%(i+1)      # 정상 데이터 파일 경로
    temp_path2 = './SplittedData/Abnormal1_%d.csv'%(i+1)   # 고장1 데이터 파일 경로
    temp_path3 = './SplittedData/Abnormal2_%d.csv'%(i+1)   # 고장2 데이터 파일 경로
    
    temp_data1 = np.array(pd.read_csv(temp_path1 , sep=',', header=None)) # 정상  데이터
    temp_data2 = np.array(pd.read_csv(temp_path2 , sep=',', header=None)) # 고장1 데이터
    temp_data3 = np.array(pd.read_csv(temp_path3 , sep=',', header=None)) # 고장2 데이터
    
    
    
    
    
    
    for j in range(NoOfSensor):
        
        for k in np.arange(select):
            
            
            
            
            
            # Normal Frequency Domain Feature
            
            
            
            
            
            
            
            
            
            
            
            # Abnormal Frequency Domain Feature
            
            
            
            
            
            
            
            
            
            
            
            # Abnormal2 Frequency Domain Feature
            
            
            
            
            
            
            
            
            
            
            
            
            

In [ ]:
# 주파수영역 특징 합치기(가로 방향)



In [ ]:
# 시간/주파수영역 특징 합치기(세로 방향)




### 2단계 결과물 제출용 파일로 저장(수강생 번호 외 코드 수정 X)

In [ ]:
StudentNo = 0   # 수강생 번호 입력

Path1 = './Result/ST%d_DC2.csv'%StudentNo
FeatureData.to_csv(Path1, header=None, index=None)

.

.

.

# [3단계] 특징 데이터 ANOVA 수행
> ### 특징 데이터 분리하여 정상/고장1/고장2 각각 ANOVA 기반 구분성 상위 10개 특징 선택
> ### ANOVA: scipy.stats 라이브러리의 f_oneway 함수 사용
> ##### (참고:https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html)

## 필수!
* 정상/고장1/고장2  P value 기반 구분성 상위 10개 선택된 특징(DataFrame) 변수명 : FeatureSelected

#### 특징데이터 분리: 정상, 고장1, 고장2

In [ ]:
NoOfData = int(FeatureData.shape[1]/3)
Normal_FeatureData    = FeatureData.iloc[:,:NoOfData]
Abnormal_FeatureData1 = FeatureData.iloc[:,NoOfData:NoOfData*2]
Abnormal_FeatureData2 = FeatureData.iloc[:,NoOfData*2:]

print(Normal_FeatureData.shape)
print(Abnormal_FeatureData1.shape)
print(Abnormal_FeatureData2.shape)

#### 정상/고장1/고장2 특징 별 ANOVA 수행하여 P value 계산

In [ ]:
NoOfFeature = FeatureData.shape[0] # 추출된 Feature 갯수

P_value = np.zeros((NoOfFeature , 2))

# 특징값 각각 T-검정 수행
for i in np.arange(NoOfFeature):
    
    
    
    
    
P_value      = pd.DataFrame(P_value)
P_value

#### P value 기준 정렬

In [ ]:
P_value_Rank = 



#### P value 기반 구분성 상위 10개 특징 선택

In [ ]:
# StartRank 부터 Number 만큼의 Feature
StartRank = 1
Number    = 10

SelectedFeatures = np.zeros((Number, FeatureData.shape[1]))

s = 0

for i in range(StartRank, StartRank+Number):
    
    
    
    
    
    



### 3단계 결과물 제출용 파일로 저장(수강생 번호 외 코드 수정 X)

In [ ]:
StudentNo = 0   # 수강생 번호 입력

Path1 = './Result/ST%d_DC3.csv'%StudentNo

FeatureSelected.to_csv(Path1, header=None, index=None)

.

.

.

# [4단계] 선택된 특징 각각 차원축소(PCA) 및 2D Plot 시각화

## 필수!
* PCA 적용 전 특징 데이터 표준화 (StandardScaler 활용)
* Plot 설정 아래와 같이 수행
               figsize = (10, 10)   
               점 색깔: 정상=파랑('b'), 고장1=빨강('r'), 고장2=자홍('m')   
               Label: 정상='Normal', 고장1='Abnormal1', 고장2='Abnormal2'
               공통 옵션: linestyle='', marker='o'
               가로/세로 축 이름 각각 PC1/PC2

* 출력된 그래프를 'ST(수강생번호)_DC4.png'이름으로 Result 폴더 내 저장

In [ ]:
FeatureSelected = 

In [ ]:
# 선택된 특징데이터에 대한 표준화
FeatureSelected_normed = 

#### 선정된 특징 PCA 통한 차원축소

In [ ]:
# 10개 PC(Principal Component) 추출
pca = PCA(n_components = 2)
PC  = pca.fit_transform(FeatureSelected_normed)

In [ ]:
# PC1, PC2 2가지 주성분의 산점도 그리기















.

.

.

# ● 결과가 저장된 폴더(Result)와 작성 완료된 본 코드 파일을 하나의 zip파일로 제출
> ## 압축파일 이름 ST(수강생번호)_DC (예시: 'ST00_DC', 'ST0_DC')
압축파일 더블클릭시 Result 폴더, DataChallenge_ST-(수강생번호) 나타나도록 파일 구성